**Biomedical BERT Fine-Tuning Experiment**

The previous experiments showed that changing the loss function and improving the learning-rate schedule produced only modest improvements, while the model continued to overfit after approximately 1–2 epochs.

This experiment tests a different hypothesis: whether the pretrained language representation itself is limiting performance.

Instead of bert-base-uncased, we will use BiomedBERT (formerly PubMedBERT, abstracts-only), which was pretrained specifically on biomedical PubMed abstracts. This provides the model with domain-specific language representations before task-specific fine-tuning.

To isolate the effect of domain-specific pretraining, all major training settings will remain unchanged from our best BERT-base experiment:

Unweighted Cross-Entropy Loss
AdamW optimizer
Learning rate: 2e-5
Weight decay: 0.01
10% linear warm-up
Linear learning-rate decay
Gradient clipping: max_norm=1.0
Batch size: 8
Maximum sequence length: 512
Maximum epochs: 8
Early stopping patience: 2
Model selection based on validation Macro-F1

In [4]:
import torch
import transformers

print("PyTorch version:", torch.__version__)
print("Transformers version:", transformers.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

PyTorch version: 2.11.0+cu128
Transformers version: 5.16.1
CUDA available: True
GPU: Tesla T4
Device: cuda


In [5]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

In [6]:
#Dataset
from datasets import load_dataset

dataset = load_dataset(
    "TimSchopf/medical_abstracts"
)

print(dataset)

README.md:   0%|          | 0.00/4.96k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.67MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.94MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11550 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2888 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['condition_label', 'medical_abstract'],
        num_rows: 11550
    })
    test: Dataset({
        features: ['condition_label', 'medical_abstract'],
        num_rows: 2888
    })
})


In [7]:
#To pandas
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

print("Original train:", len(train_df))
print("Test:", len(test_df))

Original train: 11550
Test: 2888


In [8]:
#Train/Validation split
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    train_df,
    test_size=0.15,
    stratify=train_df["condition_label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("Training samples  :", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples      :", len(test_df))

Training samples  : 9817
Validation samples: 1733
Test samples      : 2888


In [9]:
#Bio-Med BERT
MODEL_NAME = (
    "microsoft/"
    "BiomedNLP-BiomedBERT-base-uncased-abstract"
)

MAX_LENGTH = 512
NUM_CLASSES = 5

print("Model:", MODEL_NAME)
print("Maximum sequence length:", MAX_LENGTH)
print("Number of classes:", NUM_CLASSES)

Model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Maximum sequence length: 512
Number of classes: 5


In [10]:
#Load Bio-Medical Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/225k [00:00<?, ?B/s]

Tokenizer loaded successfully.
Vocabulary size: 28895


In [11]:
#Dataset Class
class MedicalAbstractDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length=512
    ):

        self.texts = dataframe[
            "medical_abstract"
        ].tolist()

        self.labels = (
            dataframe["condition_label"].values - 1
        ).astype(np.int64)

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding[
                "input_ids"
            ].squeeze(0),

            "attention_mask": encoding[
                "attention_mask"
            ].squeeze(0),

            "labels": torch.tensor(
                label,
                dtype=torch.long
            )
        }

In [12]:
#Creating Dataset
train_dataset = MedicalAbstractDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = MedicalAbstractDataset(
    val_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = MedicalAbstractDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 9817
Validation: 1733
Test: 2888


In [13]:
#Dataloaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Batch size: 8
Train batches: 1228
Validation batches: 217
Test batches: 361


In [14]:
#Batch Verification
batch = next(iter(train_loader))

print(
    "Input IDs shape     :",
    batch["input_ids"].shape
)

print(
    "Attention mask shape:",
    batch["attention_mask"].shape
)

print(
    "Labels shape        :",
    batch["labels"].shape
)

print(
    "Labels:",
    batch["labels"]
)

Input IDs shape     : torch.Size([8, 512])
Attention mask shape: torch.Size([8, 512])
Labels shape        : torch.Size([8])
Labels: tensor([4, 0, 2, 1, 3, 3, 0, 4])


In [15]:
#Define the Model
class MedicalBiomedBERTClassifier(nn.Module):

    def __init__(
        self,
        model_name,
        num_classes=5
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name
        )

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_classes
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_embedding = (
            outputs.last_hidden_state[:, 0, :]
        )

        logits = self.classifier(
            cls_embedding
        )

        return logits

In [16]:
#creating Fresh Model
model_biomed = MedicalBiomedBERTClassifier(
    model_name=MODEL_NAME,
    num_classes=NUM_CLASSES
).to(device)

print("Fresh BiomedBERT model loaded.")

print(
    "Model device:",
    next(model_biomed.parameters()).device
)

print(
    "Hidden size:",
    model_biomed.bert.config.hidden_size
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fresh BiomedBERT model loaded.
Model device: cuda:0
Hidden size: 768


In [17]:
#Loss
criterion_biomed = nn.CrossEntropyLoss()

print(criterion_biomed)

CrossEntropyLoss()


In [18]:
#AdamW optimizer
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

optimizer_biomed = AdamW(
    model_biomed.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

Learning rate: 2e-05
Weight decay: 0.01


In [19]:
#Scheduler+Warmup
NUM_EPOCHS = 8

steps_per_epoch = len(train_loader)

total_training_steps = (
    steps_per_epoch * NUM_EPOCHS
)

warmup_steps = int(
    0.10 * total_training_steps
)

scheduler_biomed = get_linear_schedule_with_warmup(
    optimizer_biomed,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

print("Steps per epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Warm-up steps:", warmup_steps)

Steps per epoch: 1228
Total training steps: 9824
Warm-up steps: 982


In [20]:
#Training Function
def train_one_epoch(
    model,
    dataloader,
    criterion,
    optimizer,
    scheduler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )

        attention_mask = batch[
            "attention_mask"
        ].to(
            device,
            non_blocking=True
        )

        labels = batch["labels"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(
            logits,
            labels
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        scheduler.step()

        running_loss += (
            loss.item() * labels.size(0)
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            (predictions == labels)
            .sum()
            .item()
        )

        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total

    return epoch_loss, epoch_accuracy

In [21]:
#Validation Function
def evaluate_model(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch[
                "input_ids"
            ].to(
                device,
                non_blocking=True
            )

            attention_mask = batch[
                "attention_mask"
            ].to(
                device,
                non_blocking=True
            )

            labels = batch[
                "labels"
            ].to(
                device,
                non_blocking=True
            )

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(
                logits,
                labels
            )

            running_loss += (
                loss.item() * labels.size(0)
            )

            predictions = torch.argmax(
                logits,
                dim=1
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    total = len(all_labels)

    epoch_loss = running_loss / total

    epoch_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )

In [22]:
#Train Bio-Bert Model
PATIENCE = 2

best_val_macro_f1 = -float("inf")
patience_counter = 0

for epoch in range(NUM_EPOCHS):

    train_loss, train_accuracy = train_one_epoch(
        model_biomed,
        train_loader,
        criterion_biomed,
        optimizer_biomed,
        scheduler_biomed,
        device
    )

    (
        val_loss,
        val_accuracy,
        val_macro_f1
    ) = evaluate_model(
        model_biomed,
        val_loader,
        criterion_biomed,
        device
    )

    current_lr = (
        optimizer_biomed
        .param_groups[0]["lr"]
    )

    print(f"\nEpoch [{epoch + 1}/{NUM_EPOCHS}]")
    print("-" * 60)

    print(
        f"Train Loss     : {train_loss:.4f}"
    )

    print(
        f"Train Accuracy : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Val Loss       : {val_loss:.4f}"
    )

    print(
        f"Val Accuracy   : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Val Macro-F1   : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Learning Rate  : "
        f"{current_lr:.8f}"
    )

    if val_macro_f1 > best_val_macro_f1:

        best_val_macro_f1 = val_macro_f1
        patience_counter = 0

        torch.save(
            model_biomed.state_dict(),
            "best_biomedbert.pt"
        )

        print(
            "✓ Best BiomedBERT model saved."
        )

    else:

        patience_counter += 1

        print(
            f"No improvement. "
            f"Patience: "
            f"{patience_counter}/{PATIENCE}"
        )

        if patience_counter >= PATIENCE:

            print(
                "\nEarly stopping triggered."
            )

            break

KeyboardInterrupt: 

In [23]:
#Validation error analysis
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import torch

# Load the best BiomedBERT checkpoint
model_biomed.load_state_dict(
    torch.load("best_biomedbert.pt", map_location=device)
)

model_biomed.to(device)
model_biomed.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:

        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        logits = model_biomed(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


# Class names
class_names = [
    "Neoplasms",
    "Digestive system diseases",
    "Nervous system diseases",
    "Cardiovascular diseases",
    "General pathological conditions"
]


# Classification report
print("BiomedBERT Validation Classification Report")
print("=" * 60)

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)


# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

print("\nConfusion Matrix:")
print(cm)

FileNotFoundError: [Errno 2] No such file or directory: 'best_biomedbert.pt'

In [24]:
import os

print(os.path.exists("best_biomedbert.pt"))

False
